# Task 5 — Source Metadata Ingestion into MongoDB

## Approach and reasoning

A Spark Structured Streaming job consumes `cpg.metadata` and writes through the
MongoDB Spark Connector into `cpg.file_metadata`.

Two design decisions carry the lab's requirements:

**1. Checkpointing.** `checkpointLocation` makes Spark commit Kafka offsets
transactionally with the batch. On restart the job resumes from the last
committed offset instead of reprocessing the topic from the beginning — this is
what "skips already-processed offsets for unchanged files" means in practice.

**2. Upsert instead of append.** A plain `append` write inserts a new document
every time a file is reprocessed, which would fail Task 6. Inside `foreachBatch`
we write with `operationType=update` and `idFieldList=file_id`, so a reprocessed
file **updates its single document in place**.

`foreachBatch` was chosen over a direct streaming sink because it hands us a
static DataFrame per micro-batch, where the connector's full upsert semantics are
available.

### Step 1 — start the streaming job

Run this in a **separate terminal** and leave it running; it is a long-lived
process, not a notebook cell.

```bash
spark-submit \
  --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,org.mongodb.spark:mongo-spark-connector_2.12:10.4.0 \
  src/spark/spark_mongo_stream.py \
  --bootstrap localhost:9092 \
  --mongo-uri mongodb://localhost:27017 \
  --checkpoint file:///tmp/chk/cpg_metadata
```

Paste the batch log output below as evidence:

```
batch 0: upserted 59 metadata docs
```

### Step 2 — verify the documents landed in MongoDB

In [1]:
import os
os.chdir("..")
!docker exec mongodb mongosh --quiet cpg --eval "db.file_metadata.countDocuments()"

59


### Step 3 — inspect one document

In [2]:
!docker exec mongodb mongosh --quiet cpg --eval \
  "printjson(db.file_metadata.findOne({rel_path: 'optimum/version.py'}))"

{
  _id: ObjectId('6a61bedeea4dfbfb3c244a8a'),
  file_id: 'b3fc55405381fc8a',
  edge_counts: {
    AST: 18,
    CFG: 3,
    DFG: 2
  },
  event_time: '2026-07-23T08:00:44.393737+00:00',
  event_type: 'metadata',
  file_hash: 'f5b8514ad074172d89b9d04fecc61d62654eb7c4b7a5dbe743b46d5eb515c56f',
  imports: [],
  loc: 20,
  num_ast_nodes: 19,
  num_classes: 0,
  num_edges: 23,
  num_functions: 1,
  rel_path: 'optimum/version.py',
  schema_version: '1.0.0'
}


### Step 4 — the duplicate check that must return an empty array

In [3]:
!docker exec mongodb mongosh --quiet cpg --eval \
  "printjson(db.file_metadata.aggregate([{\$group:{_id:'\$file_id',c:{\$sum:1}}},{\$match:{c:{\$gt:1}}}]).toArray())"

[]


### Step 5 — inspect the checkpoint directory Spark maintains

In [4]:
!ls -R /tmp/chk/cpg_metadata 2>/dev/null | head -20
!cat /tmp/chk/cpg_metadata/offsets/0 2>/dev/null | tail -1

/tmp/chk/cpg_metadata:
commits
metadata
offsets
sources

/tmp/chk/cpg_metadata/commits:
0
1
10
11
12
13
14
15
16
17
18
19
2


{"cpg.metadata":{"0":59}}

### Database UI evidence

Connect MongoDB Compass to `mongodb://localhost:27017` and open
`cpg.file_metadata`.

Screenshots captured on this machine:

![MongoDB Compass collection](images/mongo_collection.png)

![Spark UI streaming query](images/spark_ui.png)

## Reflection

**What happened on this machine.** Only ~2–3 GB of RAM were free, so the Spark
job was deliberately started *after* the Neo4j ingestion finished — nothing was
lost, because the job reads `cpg.metadata` from the beginning: its first
micro-batch upserted all 59 documents (`batch 0: upserted 59 metadata docs`).
During the Task 6 replay it resumed from the committed checkpoint offset and
processed exactly one message (`batch 1: upserted 1 metadata docs`), which is
the checkpoint-recovery evidence the lab asks for.

**What worked.** `foreachBatch` with an upsert keyed on `rel_path`: the
replayed file kept the same Mongo `_id` and the collection stayed at 59
documents. Passing the checkpoint as an explicit `file:///tmp/chk/cpg_metadata`
URI avoided the `fs.defaultFS`/HDFS trap (TROUBLESHOOTING.md §10), where a bare
path on a machine with Hadoop configured resolves to `hdfs://` and dies in
`org.apache.hadoop.ipc.Client` with Connection refused. Declaring an explicit
`StructType` also doubles as executable documentation of the Task 3 message
contract — schema inference is unavailable on streaming sources anyway.

**Version pinning.** The `--packages` coordinates must match the Spark line and
Scala version exactly; a mismatch produces `ClassNotFoundException` at submit
time rather than a clear error.
